In [89]:
import pandas as pd
import numpy as np

In [90]:
clean_df = pd.read_csv("../data/clean_events.csv")

clean_df['event_time'] = pd.to_datetime(
    clean_df['event_time']
)

In [91]:
clean_df = clean_df.sort_values(
    ['user_id', 'event_time']
).reset_index(drop=True)

In [92]:
clean_df['previous_event_time'] = (
    clean_df.groupby('user_id')['event_time'].shift(1)
)

In [93]:
clean_df['time_gap_minutes'] = (
    clean_df['event_time']
    - clean_df['previous_event_time']
).dt.total_seconds() / 60

In [94]:
clean_df['new_session'] = (
    clean_df['time_gap_minutes'].isna()
    | (clean_df['time_gap_minutes'] > 30)
)

In [95]:
clean_df['session_number'] = (
    clean_df.groupby('user_id')['new_session']
    .cumsum()
)

In [96]:
clean_df['reconstructed_session_id'] = (
    clean_df['user_id'].astype(str)
    + '_'
    + clean_df['session_number'].astype(str)
)

In [172]:
clean_df.to_csv(
    "../data/reconstructed_events.csv",
    index=False
)

In [9]:
clean_df['new_session'] = (
    clean_df['time_gap_minutes'].isna() |
    (clean_df['time_gap_minutes'] > 30)
)

In [97]:
clean_df[
    [
        'user_id',
        'event_time',
        'new_session',
        'session_number',
        'reconstructed_session_id'
    ]
].head(20)

,user_id,event_time,new_session,session_number,reconstructed_session_id
0,1515915625353226922,2020-10-29 11:28:35+00:00,True,1,1515915625353226922_1
1,1515915625353230067,2020-10-06 06:30:32+00:00,True,1,1515915625353230067_1
2,1515915625353230683,2020-11-09 08:52:51+00:00,True,1,1515915625353230683_1
3,1515915625353230683,2020-11-09 09:04:34+00:00,False,1,1515915625353230683_1
4,1515915625353230683,2020-11-09 09:08:53+00:00,False,1,1515915625353230683_1
5,1515915625353230683,2020-11-09 09:10:59+00:00,False,1,1515915625353230683_1
6,1515915625353230683,2020-11-09 09:15:27+00:00,False,1,1515915625353230683_1
7,1515915625353230683,2020-11-09 09:22:00+00:00,False,1,1515915625353230683_1
8,1515915625353230683,2020-11-09 09:25:34+00:00,False,1,1515915625353230683_1
9,1515915625353230683,2020-11-09 09:26:38+00:00,False,1,1515915625353230683_1


In [98]:
clean_df['reconstructed_session_id'].nunique()

504630

In [99]:
clean_df['reconstructed_session_id'].duplicated().sum()

np.int64(379682)

In [100]:
session_features = clean_df.groupby(
    'reconstructed_session_id'
).agg(
    session_start=('event_time', 'min'),
    session_end=('event_time', 'max')
).reset_index()

In [101]:
session_features['session_duration_minutes'] = (
    session_features['session_end']
    - session_features['session_start']
).dt.total_seconds() / 60

In [102]:
session_features.shape

(504630, 4)

In [105]:
view_counts = (
    clean_df[clean_df['event_type'] == 'view']
    .groupby('reconstructed_session_id')
    .size()
    .reset_index(name='num_views_in_session')
)

In [107]:
session_features.columns.tolist()

['reconstructed_session_id',
 'session_start',
 'session_end',
 'session_duration_minutes']

In [108]:
view_counts.columns.tolist()

['reconstructed_session_id', 'num_views_in_session']

In [109]:
view_counts.head()

,reconstructed_session_id,num_views_in_session
0,1515915625353226922_1,1
1,1515915625353230067_1,1
2,1515915625353230683_1,8
3,1515915625353230683_2,4
4,1515915625353230683_3,1


In [110]:
session_features = session_features.merge(
    view_counts,
    on='reconstructed_session_id',
    how='left'
)

In [111]:
session_features.columns.tolist()

['reconstructed_session_id',
 'session_start',
 'session_end',
 'session_duration_minutes',
 'num_views_in_session']

In [112]:
session_features['num_views_in_session'] = (
    session_features['num_views_in_session']
    .fillna(0)
    .astype(int)
)

In [113]:
session_features[
    [
        'reconstructed_session_id',
        'session_duration_minutes',
        'num_views_in_session'
    ]
].head(10)

,reconstructed_session_id,session_duration_minutes,num_views_in_session
0,1515915625353226922_1,0.000000,1
1,1515915625353230067_1,0.000000,1
2,1515915625353230683_1,33.783333,8
3,1515915625353230683_2,7.850000,4
4,1515915625353230683_3,0.000000,1
5,1515915625353230922_1,0.000000,1
6,1515915625353234047_1,0.666667,2
7,1515915625353234047_10,0.100000,2
8,1515915625353234047_2,0.000000,1
9,1515915625353234047_3,4.550000,6


In [114]:
session_features['num_views_in_session'].describe()

count    504630.000000
mean          1.571335
std           1.672999
min           0.000000
25%           1.000000
50%           1.000000
75%           2.000000
max         260.000000
Name: num_views_in_session, dtype: float64

In [115]:
cart_counts = (
    clean_df[clean_df['event_type'] == 'cart']
    .groupby('reconstructed_session_id')
    .size()
    .reset_index(name='num_cart_adds_in_session')
)

In [116]:
session_features = session_features.merge(
    cart_counts,
    on='reconstructed_session_id',
    how='left'
)

In [117]:
session_features['num_cart_adds_in_session'] = (
    session_features['num_cart_adds_in_session']
    .fillna(0)
    .astype(int)
)

In [118]:
session_features[
    [
        'reconstructed_session_id',
        'num_views_in_session',
        'num_cart_adds_in_session'
    ]
].head(10)

,reconstructed_session_id,num_views_in_session,num_cart_adds_in_session
0,1515915625353226922_1,1,0
1,1515915625353230067_1,1,0
2,1515915625353230683_1,8,0
3,1515915625353230683_2,4,0
4,1515915625353230683_3,1,0
5,1515915625353230922_1,1,0
6,1515915625353234047_1,2,0
7,1515915625353234047_10,2,0
8,1515915625353234047_2,1,0
9,1515915625353234047_3,6,0


In [119]:
session_features['num_cart_adds_in_session'].describe()

count    504630.000000
mean          0.107061
std           0.442846
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          25.000000
Name: num_cart_adds_in_session, dtype: float64

In [120]:
purchase_counts = (
    clean_df[clean_df['event_type'] == 'purchase']
    .groupby('reconstructed_session_id')
    .size()
    .reset_index(name='num_purchases_in_session')
)

In [121]:
session_features = session_features.merge(
    purchase_counts,
    on='reconstructed_session_id',
    how='left'
)

In [122]:
session_features['num_purchases_in_session'] = (
    session_features['num_purchases_in_session']
    .fillna(0)
    .astype(int)
)

In [123]:
session_features['purchase_flag'] = (
    session_features['num_purchases_in_session'] > 0
).astype(int)

In [124]:
session_features[
    [
        'reconstructed_session_id',
        'num_views_in_session',
        'num_cart_adds_in_session',
        'num_purchases_in_session',
        'purchase_flag'
    ]
].head(10)

,reconstructed_session_id,num_views_in_session,num_cart_adds_in_session,num_purchases_in_session,purchase_flag
0,1515915625353226922_1,1,0,0,0
1,1515915625353230067_1,1,0,0,0
2,1515915625353230683_1,8,0,0,0
3,1515915625353230683_2,4,0,0,0
4,1515915625353230683_3,1,0,0,0
5,1515915625353230922_1,1,0,0,0
6,1515915625353234047_1,2,0,0,0
7,1515915625353234047_10,2,0,0,0
8,1515915625353234047_2,1,0,0,0
9,1515915625353234047_3,6,0,0,0


In [125]:
session_features['purchase_flag'].value_counts()

purchase_flag
0    480124
1     24506
Name: count, dtype: int64

In [126]:
session_features['cart_session_flag'] = (
    session_features['num_cart_adds_in_session'] > 0
).astype(int)

In [127]:
session_features['cart_session_flag'].value_counts()

cart_session_flag
0    461777
1     42853
Name: count, dtype: int64

In [129]:
session_features[
    (session_features['purchase_flag'] == 1) &
    (session_features['cart_session_flag'] == 0)
].shape

(2453, 9)

In [130]:
cart_sessions = session_features[
    session_features['cart_session_flag'] == 1
].copy()

In [131]:
cart_sessions['is_abandoned'] = (
    cart_sessions['purchase_flag'] == 0
).astype(int)

In [132]:
cart_sessions['is_abandoned'].value_counts()

is_abandoned
0    22053
1    20800
Name: count, dtype: int64

In [133]:
first_cart_time = (
    clean_df[clean_df['event_type'] == 'cart']
    .groupby('reconstructed_session_id')['event_time']
    .min()
    .reset_index()
)

first_cart_time.columns = [
    'reconstructed_session_id',
    'cart_time'
]

In [134]:
cart_sessions = cart_sessions.merge(
    first_cart_time,
    on='reconstructed_session_id',
    how='left'
)

In [135]:
cart_sessions[
    [
        'reconstructed_session_id',
        'session_start',
        'cart_time',
        'session_duration_minutes',
        'is_abandoned'
    ]
].head(10)

,reconstructed_session_id,session_start,cart_time,session_duration_minutes,is_abandoned
0,1515915625353286099_1,2020-10-03 11:20:33+00:00,2020-10-03 11:23:00+00:00,3.183333,0
1,1515915625353286099_2,2020-10-06 08:26:55+00:00,2020-10-06 08:28:18+00:00,1.383333,1
2,1515915625353457259_1,2020-09-29 05:51:33+00:00,2020-09-29 05:51:58+00:00,1.366667,0
3,1515915625353534622_2,2020-10-06 09:45:38+00:00,2020-10-06 09:45:38+00:00,44.116667,0
4,1515915625353561691_1,2020-11-24 17:34:42+00:00,2020-11-24 17:35:50+00:00,2.666667,0
5,1515915625353561691_4,2020-11-25 09:24:15+00:00,2020-11-25 09:24:15+00:00,2.733333,0
6,1515915625353706476_1,2020-12-19 18:42:56+00:00,2020-12-19 18:44:12+00:00,16.350000,1
7,1515915625353900095_10,2020-10-15 05:03:23+00:00,2020-10-15 05:09:13+00:00,6.200000,0
8,1515915625354061413_3,2020-10-29 17:22:44+00:00,2020-10-29 17:23:50+00:00,3.050000,1
9,1515915625354176736_1,2020-09-29 10:59:30+00:00,2020-09-29 11:26:29+00:00,31.683333,0


In [136]:
before_cart = clean_df.merge(
    first_cart_time,
    on='reconstructed_session_id',
    how='inner'
)

before_cart = before_cart[
    before_cart['event_time'] <= before_cart['cart_time']
].copy()

In [137]:
views_before_cart = (
    before_cart[
        before_cart['event_type'] == 'view'
    ]
    .groupby('reconstructed_session_id')
    .size()
    .reset_index(name='views_before_cart')
)

In [138]:
cart_value = (
    before_cart[
        before_cart['event_type'] == 'cart'
    ]
    .groupby('reconstructed_session_id')['price']
    .sum()
    .reset_index(name='cart_value')
)

In [139]:
duration_before_cart = (
    before_cart
    .groupby('reconstructed_session_id')['event_time']
    .agg(['min', 'max'])
    .reset_index()
)

duration_before_cart['duration_before_cart'] = (
    duration_before_cart['max']
    - duration_before_cart['min']
).dt.total_seconds() / 60

In [140]:
category_before_cart = (
    before_cart
    .groupby('reconstructed_session_id')['category_code']
    .nunique()
    .reset_index(name='category_diversity_before_cart')
)

In [141]:
avg_price_before_cart = (
    before_cart
    .groupby('reconstructed_session_id')['price']
    .mean()
    .reset_index(name='avg_price_before_cart')
)

In [142]:
price_range_before_cart = (
    before_cart
    .groupby('reconstructed_session_id')['price']
    .agg(lambda x: x.max() - x.min())
    .reset_index(name='price_range_before_cart')
)

In [143]:
cart_sessions = cart_sessions.merge(
    views_before_cart,
    on='reconstructed_session_id',
    how='left'
)

cart_sessions = cart_sessions.merge(
    cart_value,
    on='reconstructed_session_id',
    how='left'
)

cart_sessions = cart_sessions.merge(
    duration_before_cart[
        ['reconstructed_session_id', 'duration_before_cart']
    ],
    on='reconstructed_session_id',
    how='left'
)

cart_sessions = cart_sessions.merge(
    category_before_cart,
    on='reconstructed_session_id',
    how='left'
)

cart_sessions = cart_sessions.merge(
    avg_price_before_cart,
    on='reconstructed_session_id',
    how='left'
)

cart_sessions = cart_sessions.merge(
    price_range_before_cart,
    on='reconstructed_session_id',
    how='left'
)

In [145]:
cart_sessions[
    [
        'reconstructed_session_id',
        'views_before_cart',
        'cart_value',
        'duration_before_cart',
        'category_diversity_before_cart',
        'avg_price_before_cart',
        'price_range_before_cart',
        'is_abandoned'
    ]
].head(10)

,reconstructed_session_id,views_before_cart,cart_value,duration_before_cart,category_diversity_before_cart,avg_price_before_cart,price_range_before_cart,is_abandoned
0,1515915625353286099_1,1.0,119.03,2.450000,1,119.030000,0.00,0
1,1515915625353286099_2,2.0,120.27,1.383333,1,108.746667,34.57,1
2,1515915625353457259_1,1.0,55.16,0.416667,1,55.160000,0.00,0
3,1515915625353534622_2,NaN,19.05,0.000000,1,19.050000,0.00,0
4,1515915625353561691_1,2.0,172.86,1.133333,1,172.860000,0.00,0
5,1515915625353561691_4,NaN,172.86,0.000000,1,172.860000,0.00,0
6,1515915625353706476_1,1.0,40.16,1.266667,1,40.160000,0.00,1
7,1515915625353900095_10,2.0,39.75,5.833333,1,32.533333,21.65,0
8,1515915625354061413_3,1.0,89.49,1.100000,1,77.310000,24.36,1
9,1515915625354176736_1,5.0,236.97,26.983333,1,215.183333,32.68,0


In [146]:
cart_sessions[
    [
        'views_before_cart',
        'cart_value',
        'duration_before_cart',
        'category_diversity_before_cart',
        'avg_price_before_cart',
        'price_range_before_cart'
    ]
].isnull().sum()

views_before_cart                 847
cart_value                          0
duration_before_cart                0
category_diversity_before_cart      0
avg_price_before_cart               0
price_range_before_cart             0
dtype: int64

In [147]:
cart_sessions['views_before_cart'] = (
    cart_sessions['views_before_cart']
    .fillna(0)
    .astype(int)
)

In [148]:
cart_sessions[
    [
        'views_before_cart',
        'cart_value',
        'duration_before_cart',
        'category_diversity_before_cart',
        'avg_price_before_cart',
        'price_range_before_cart'
    ]
].isnull().sum()

views_before_cart                 0
cart_value                        0
duration_before_cart              0
category_diversity_before_cart    0
avg_price_before_cart             0
price_range_before_cart           0
dtype: int64

In [149]:
brand_switches_before_cart = (
    before_cart
    .sort_values(['reconstructed_session_id', 'event_time'])
    .groupby('reconstructed_session_id')['brand']
    .apply(lambda x: x.ne(x.shift()).sum() - 1)
    .reset_index(name='brand_switch_count_before_cart')
)

In [150]:
cart_sessions = cart_sessions.merge(
    brand_switches_before_cart,
    on='reconstructed_session_id',
    how='left'
)

In [151]:
cart_sessions['brand_switch_count_before_cart'] = (
    cart_sessions['brand_switch_count_before_cart']
    .fillna(0)
    .astype(int)
)

In [153]:
clean_df = clean_df.sort_values(
    ['user_id', 'event_time']
).copy()

clean_df['purchase_before_event'] = (
    clean_df.groupby('user_id')['event_type']
    .transform(
        lambda x: (x == 'purchase').cumsum().shift(1).fillna(0)
    )
)

In [154]:
previous_purchase = (
    clean_df
    .groupby('reconstructed_session_id')
    ['purchase_before_event']
    .min()
    .reset_index(name='previous_purchase_count')
)

In [155]:
cart_sessions = cart_sessions.merge(
    previous_purchase,
    on='reconstructed_session_id',
    how='left'
)

In [156]:
cart_sessions['has_previous_purchase'] = (
    cart_sessions['previous_purchase_count'] > 0
).astype(int)

In [157]:
cart_sessions[
    [
        'views_before_cart',
        'cart_value',
        'duration_before_cart',
        'category_diversity_before_cart',
        'avg_price_before_cart',
        'price_range_before_cart',
        'brand_switch_count_before_cart',
        'previous_purchase_count',
        'has_previous_purchase'
    ]
].isnull().sum()

views_before_cart                 0
cart_value                        0
duration_before_cart              0
category_diversity_before_cart    0
avg_price_before_cart             0
price_range_before_cart           0
brand_switch_count_before_cart    0
previous_purchase_count           0
has_previous_purchase             0
dtype: int64

In [158]:
cart_sessions['cart_to_view_ratio_before_cart'] = (
    cart_sessions['num_cart_adds_in_session'] /
    cart_sessions['views_before_cart'].replace(0, 1)
)

In [159]:
cart_sessions['cart_to_view_ratio_before_cart'] = (
    1 /
    cart_sessions['views_before_cart'].replace(0, 1)
)

In [160]:
cart_sessions['no_view_before_cart'] = (
    cart_sessions['views_before_cart'] == 0
).astype(int)

In [161]:
cart_sessions['cart_hour'] = (
    cart_sessions['cart_time'].dt.hour
)

In [162]:
cart_sessions['cart_day_of_week'] = (
    cart_sessions['cart_time'].dt.dayofweek
)

In [163]:
cart_sessions['cart_is_weekend'] = (
    cart_sessions['cart_day_of_week'] >= 5
).astype(int)

In [164]:
leakage_free_features = [
    'views_before_cart',
    'cart_value',
    'duration_before_cart',
    'category_diversity_before_cart',
    'avg_price_before_cart',
    'price_range_before_cart',
    'brand_switch_count_before_cart',
    'previous_purchase_count',
    'has_previous_purchase',
    'cart_to_view_ratio_before_cart',
    'no_view_before_cart',
    'cart_hour',
    'cart_day_of_week',
    'cart_is_weekend'
]

In [167]:
model_df_v2 = cart_sessions[
    leakage_free_features + ['is_abandoned']
].copy()

In [168]:
model_df_v2.shape

(42853, 15)

In [169]:
model_df_v2.isnull().sum()

views_before_cart                 0
cart_value                        0
duration_before_cart              0
category_diversity_before_cart    0
avg_price_before_cart             0
price_range_before_cart           0
brand_switch_count_before_cart    0
previous_purchase_count           0
has_previous_purchase             0
cart_to_view_ratio_before_cart    0
no_view_before_cart               0
cart_hour                         0
cart_day_of_week                  0
cart_is_weekend                   0
is_abandoned                      0
dtype: int64

In [170]:
model_df_v2['is_abandoned'].value_counts()

is_abandoned
0    22053
1    20800
Name: count, dtype: int64

In [171]:
model_df_v2.to_csv(
    "../data/model_ready_cart_sessions_v2.csv",
    index=False
)